# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and enumerate them by their @id
if hasattr(dataset.metadata, 'record_set') and dataset.metadata.record_set:
    print("Available record sets and their fields:")
    for rs in dataset.metadata.record_set:
        print(f"- Record set: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            for field in rs['field']:
                fname = field['name'] if 'name' in field else field['@id']
                print(f"    - {fname} (id: {field['@id']})")
else:
    # fallback: print available record sets from loaded metadata
    print("No explicit recordSet entries found in metadata, loading available records to inspect schema...")
    record_sets = dataset.record_sets()
    print(f"Record sets found: {record_sets}")

    # For each record set, print the first record and keys
    for rs_id in record_sets:
        print(f"\nRecord set: {rs_id}")
        try:
            records = list(dataset.records(record_set=rs_id))
            if records:
                df_sample = pd.DataFrame(records)
                print(f"Fields/columns (@id): {df_sample.columns.tolist()}")
                print(df_sample.head(1))
            else:
                print("No records found in this record set.")
        except Exception as e:
            print(f"  Could not load records for {rs_id}: {e}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Discover record set IDs as in overview step
record_sets = dataset.record_sets()  # list of record set @ids
dataframes = {}

for record_set in record_sets:
    print(f"Loading records for record_set={record_set}")
    records = list(dataset.records(record_set=record_set))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(df.head(2))
    else:
        print("  No records loaded.")

# For further analysis, select the first record set
main_record_set_id = record_sets[0] if record_sets else None
if main_record_set_id:
    print(f"Fields available in record set {main_record_set_id}: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
# Example: Assume 'MW [kDa]' and 'Abundance_Sample1' (actual column names may vary; adjust to your field @ids)
df = dataframes[main_record_set_id]

# Try to find a numeric field
candidate_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
if not candidate_numeric_fields:
    # Try converting some columns to numeric if possible
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='ignore')
    candidate_numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]

if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]
else:
    print("No numeric field found.")
    numeric_field = df.columns[0]

print(f"Using numeric field '{numeric_field}' for filtering and analysis.")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field]) else df
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field]):
    filtered_df = filtered_df.copy()  # Avoid SettingWithCopyWarning
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a text/categorical field (e.g., 'Description' or similar)
candidate_groups = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
group_field = candidate_groups[0] if candidate_groups else None
if group_field:
    print(f"Grouping by field '{group_field}'")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())
else:
    print("No categorical group field detected.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field if numeric
if pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

# If there is a group/categorical field, plot mean of numeric field by group
if group_field:
    top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(data=top_groups, x=group_field, y=numeric_field)
    plt.title(f'Mean {numeric_field} by {group_field} (top 10)')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using `mlcroissant`, we loaded metadata and multiple record sets referenced by `@id`.
- Numeric analysis and normalization were performed on core molecular/protein attributes.
- Data distributions and group-level summaries revealed central trends in the dataset.  
- For your own work, review the field `@id`s and adapt analysis/grouping to your research questions.

Continue exploring more features, record sets, and data fields as needed for your downstream ML or analysis tasks.